## Customer Segmentation

### Objective

Customer Segmentation is the process of grouping users with similar behavioral patterns into meaningful clusters.

The primary objectives of this notebook are:

- Analyze customer behavior patterns
- Identify distinct customer groups
- Build customer segments using machine learning
- Enable targeted recommendations
- Support personalization strategies

This notebook utilizes K-Means Clustering to create behavioral customer segments based on engineered user features.

The generated customer segments will be used by the Recommendation Engine and Personalization Engine in subsequent stages of the project.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## Step 1: Load Engineered Features

Load customer-level behavioral features generated during the Feature Engineering phase.

In [ ]:
import os

print(os.getcwd())

In [ ]:
customer_features = pd.read_csv(
    "../data/features/customer_features.csv"
)

customer_features.head()

In [ ]:
customer_features.info()

In [ ]:
#descriptive stats
customer_features.describe()

In [ ]:
customer_features.shape

In [ ]:
#Missing Value Validation
customer_features.isnull().sum()

## Log transformation

In [ ]:
import numpy as np

customer_features["total_interactions"] = np.log1p(
    customer_features["total_interactions"]
)

customer_features["total_views"] = np.log1p(
    customer_features["total_views"]
)

customer_features["total_cart"] = np.log1p(
    customer_features["total_cart"]
)

customer_features["total_transactions"] = np.log1p(
    customer_features["total_transactions"]
)

customer_features["unique_items"] = np.log1p(
    customer_features["unique_items"]
)

The customer behavior features were highly right-skewed, with a small number of power users generating thousands of interactions while most users had only one or two interactions. To reduce the influence of outliers on K-Means clustering, I applied a log transformation before standardization and clustering

## Feature Selection

Select numerical behavioral features for clustering

In [ ]:
X = customer_features.drop(
    columns=["visitorid"]
)

## Feature Scaling

Standardization ensures that features with larger magnitudes do not dominate clustering.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

## Elbow Method

Determine the optimal number of clusters.

In [ ]:
inertia = [] #dist from centre to all data points

k_values = range(2, 11)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(model.inertia_)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    k_values, #x-axis
    inertia,#y-axis
    marker="o"
)

plt.xlabel("Number of Clusters")

plt.ylabel("Inertia")

plt.title("Elbow Method")

plt.show()

In [ ]:
feature_columns = [
    "total_interactions",
    "total_views",
    "total_cart",
    "total_transactions",
    "unique_items",
    "avg_interaction_strength",
    "recency_days"
]

In [ ]:
customer_features.columns

## Silhouette Analysis

Evaluate clustering quality for different values of K.

In [ ]:
sample_df = customer_features.head(10000)

X_sample = scaler.fit_transform(
    sample_df[feature_columns]
)

In [ ]:
scores = []

for k in range(5,6,7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_sample)

    score = silhouette_score(
        X_sample,
        labels
    )

    scores.append(score)

    print(
        f"K={k}, Silhouette Score={score:.4f}"
    )

In [ ]:
#train final model
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

customer_features["cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
#cluster distribution
customer_features["cluster"].value_counts()

In [ ]:
customer_features[feature_columns].describe()

In [ ]:
#cluster profiling 
cluster_profile = (
    customer_features
    .groupby("cluster")
    .mean()
)

cluster_profile

In [ ]:
cluster_profile = cluster_profile.drop('visitorid',axis=1)

In [ ]:
#heatmap
import seaborn as sns
plt.figure(figsize=(12,6))

sns.heatmap(
    cluster_profile,
    annot=True,
    cmap="Blues"
)

plt.title(
    "Cluster Profile Heatmap"
)

plt.show()

## Assign Segment Names

In [ ]:
cluster_names = {
    0: "Browsers",
    1: "Inactive",
    2: "Regular Users",
    3: "Buyers",
    4: "Cart Users"
}


customer_features["segment"] = (
    customer_features["cluster"]
    .map(cluster_names)
)

In [ ]:
customer_features.head()

## Segment Distribution

In [ ]:
customer_features["segment"].value_counts()

In [ ]:
#Segment Summary
customer_features.groupby(
    "segment"
).mean()

## Saving Segmentation Output

In [ ]:
customer_features.to_csv(
    "../data/features/customer_segments.csv",
    index=False
)

## Pipeline

In [ ]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
segmentation_pipeline = Pipeline(
    [   #step1 : log transformation
        (
            "log_transform",
            FunctionTransformer(
                np.log1p,
                validate=False
            )
        ),
         #step2 : scaling
        (
            "scaler",
            StandardScaler()
        ),
        #step3 : clustering
        (
            "kmeans",
            KMeans(
                n_clusters=5,
                random_state=42,
                n_init=10
            )
        )
    ]
)

In [ ]:
from sklearn import set_config

set_config(display="diagram")

segmentation_pipeline

## Training Pipeline

In [ ]:
customer_features["cluster"] = (
    segmentation_pipeline.fit_predict(
        customer_features[feature_columns]
    )
)

In [ ]:
print(segmentation_pipeline)

## Save Pipeline

In [ ]:
import joblib

joblib.dump(
    segmentation_pipeline,
    "../models/segmentation_pipeline.pkl"
)

Conclusion
In this notebook, customer behavioral features were standardized and grouped into meaningful segments using K-Means clustering.

Completed Tasks:

Loaded engineered customer features from csv
Standardized numerical variables
Determined the optimal number of clusters
Applied K-Means clustering
Profiled customer groups
Assigned business-friendly segment labels
Saved customer segmentation results
Generated Output:

customer_segments.csv
segmentation_pipeline.pkl
Business Impact:

Customer segmentation enables targeted marketing, personalized recommendations, and customer profiling.